# Data Quality untuk AI dengan Great Expectations
**Hands-on 60 menit.** Semua data sintetis. Instalasi dilakukan sebelum kelas.
Fokus: profiling, aturan executable, root cause join, dan verifikasi perbaikan.
Baca DATA_DICTIONARY.md. Nilai dan threshold merupakan simulasi, bukan standar industri.


## 00–05 menit: konteks dan setup
Prediksi churn pada 1 Agustus 2026, label 30 hari ke depan sudah matang pada tanggal observasi.
Satu baris = satu pelanggan pada satu tanggal snapshot. ID adalah string, nol awal bermakna.
Jalankan sel setup. Tidak ada instalasi otomatis dalam notebook.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import great_expectations as gx
from IPython.display import display, HTML

ROOT = Path.cwd()
if not (ROOT / 'src' / 'lab.py').exists() and (ROOT.parent / 'src' / 'lab.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'lab.py').exists(), 'Jalankan dari folder paket.'
sys.path.insert(0, str(ROOT))
from src.lab import load_data, profile, create_checkpoint, validate, build_snapshot
assert gx.__version__ == '1.24.0', 'Pilih kernel sesuai requirements.txt'
print('GX:', gx.__version__, 'pandas:', pd.__version__)
df, customers, usage = load_data(ROOT)
display(df.head())


## 05–15 menit: profiling
Jalankan ringkasan. Lengkapi kolom yang ingin diperiksa. Catat:
(1) jumlah baris, (2) missing usage, (3) duplikasi kunci gabungan, (4) biaya negatif.
Bedakan jumlah baris duplikat tambahan dengan seluruh baris yang masuk kelompok duplikat.


In [ ]:
display(profile(df))
# TODO: ikuti unit satu baris di data dictionary.
key_columns = [...]  # dua nama kolom
target_column = ...  # feature usage
assert Ellipsis not in key_columns and target_column is not Ellipsis, 'Lengkapi TODO profiling.'


In [ ]:
metrics = pd.Series({
    'rows': len(df),
    'missing_id': df['customer_id'].isna().sum(),
    'missing_usage': df[target_column].isna().sum(),
    'missing_usage_percent': round(df[target_column].isna().mean()*100, 2),
    'duplicate_extra_rows': df.duplicated(key_columns).sum(),
    'all_rows_in_duplicate_groups': df.duplicated(key_columns, keep=False).sum(),
    'negative_fee': (df['monthly_fee'] < 0).sum(),
})
display(metrics)
display(df.groupby('source_system')['usage_30d'].agg(
    rows='size', available='count'))


**Catatan peserta:** temuan mana paling berisiko? Apakah missing berarti nol penggunaan?
Tuliskan observasi, belum kesimpulan root cause, pada LAPORAN_PESERTA.md.

## 15–30 menit: empat Expectations dan Checkpoint
GX menggunakan Data Source, Data Asset, Batch Definition, Expectation Suite,
Validation Definition, dan Checkpoint. Kerangka tersebut sudah tersedia di src/lab.py.
Lengkapi parameter aturan berdasarkan kontrak latihan. `mostly=0.95` berarti minimal
95% baris memenuhi aturan, bukan 95% nilai harus kosong.
`meta` mencatat owner dan kebijakan; keputusan HOLD dibuat eksplisit setelah validasi.


In [ ]:
# TODO: lengkapi empat parameter dari DATA_DICTIONARY.md.
ID_COLUMN = ...
UNIQUE_COLUMNS = [...]
FEE_MIN = ...
USAGE_MOSTLY = ...
assert ID_COLUMN is not Ellipsis and Ellipsis not in UNIQUE_COLUMNS, 'Lengkapi aturan ID.'
assert FEE_MIN is not Ellipsis and USAGE_MOSTLY is not Ellipsis, 'Lengkapi aturan nilai dan threshold.'


In [ ]:
def rule_meta(rule_id, owner, action):
    return dict(rule_id=rule_id, owner=owner,
        severity_policy='critical', action_on_failure=action)

expectations = [
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column=ID_COLUMN,
        meta=rule_meta('DQ01', 'Data Owner pelanggan', 'hold; investigasi ID kosong')),
    gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=UNIQUE_COLUMNS, ignore_row_if='never',
        meta=rule_meta('DQ02', 'Data Steward pelanggan', 'hold; investigasi replay')),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='monthly_fee', min_value=FEE_MIN,
        meta=rule_meta('DQ03', 'Data Owner billing', 'hold; konfirmasi billing')),
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column='usage_30d', mostly=USAGE_MOSTLY,
        meta=rule_meta('DQ04', 'Data Steward usage', 'hold; investigasi join')),
]
context, checkpoint = create_checkpoint(ROOT, expectations)
before, docs_before = validate(context, checkpoint, df, 'before', ROOT)
display(before)
for url in docs_before:
    print('Buka Data Docs di browser:', url)
    display(HTML(f'<a href="{url}" target="_blank">Data Docs lokal</a>'))


## 30–40 menit: investigasi root cause
Buka Data Docs. Pilih run `before` dan periksa rule DQ04. Catat unexpected count.
Sel berikut membandingkan source, hasil normalisasi, dan pasangan join.
Cari bukti pada data, bukan hanya membaca nama fungsi.


In [ ]:
from src.lab import legacy_normalize
trace = customers[['customer_id', 'snapshot_date', 'source_system']].copy()
trace = trace.rename(columns={'customer_id': 'source_customer_id'})
trace['transformed_customer_id'] = legacy_normalize(customers)
trace = trace.merge(
    usage[['customer_id', 'snapshot_date', 'usage_30d']].rename(
        columns={'customer_id':'transformed_customer_id'}),
    on=['transformed_customer_id', 'snapshot_date'], how='left',
    validate='many_to_one', indicator=True)
display(trace.head(25))
print('Missing usage di sumber:', usage['usage_30d'].isna().sum())
display(trace.groupby('source_system')['_merge'].value_counts())


**Jawab sebelum memperbaiki:**
1. Apakah usage hilang pada sumber atau setelah transformasi?
2. Apa perbedaan ID sumber dan ID hasil transformasi?
3. Mengapa masalah terkonsentrasi pada satu source_system?
4. Apa penyebab missing yang masih tersisa di luar kelompok tersebut?

## 40–50 menit: koreksi dan validasi ulang
Perbaiki normalisasi berdasarkan sumber resmi. Jangan mengisi usage dengan nol,
menurunkan threshold, atau menghapus baris demi meluluskan aturan.
Jangan memperbaiki biaya/duplikasi sekaligus: kita ingin mengisolasi efek perbaikan join.


In [ ]:
def normalize_id_fixed(customers):
    # TODO: pertahankan identitas dari sumber resmi sebagai string.
    # Hapus whitespace jika ada. Pertahankan nilai kosong sebagai missing.
    return ...
assert normalize_id_fixed(customers) is not Ellipsis, 'Lengkapi normalisasi ID.'


In [ ]:
after_df = build_snapshot(customers, usage, normalize_id=normalize_id_fixed)
after_df.to_csv(ROOT / 'data' / 'customer_snapshot_after.csv', index=False)
after, docs_after = validate(context, checkpoint, after_df, 'after', ROOT)
comparison = before[['rule_id','success','unexpected_count']].merge(
    after[['rule_id','success','unexpected_count']], on='rule_id', suffixes=('_before','_after'))
display(comparison)
print('Baris sebelum / sesudah:', len(df), len(after_df))
print('Missing usage sebelum / sesudah:', df['usage_30d'].isna().sum(), after_df['usage_30d'].isna().sum())
display(after_df.loc[after_df['usage_30d'].isna()])
batch_status = 'RELEASE' if after['success'].all() else 'HOLD'
print('Keputusan batch:', batch_status)
print('Data Docs:', docs_after)


## 50–60 menit: laporan dan debrief
Isi LAPORAN_PESERTA.md dengan:
- Gejala, bukti sumber/transformasi, dan root cause.
- Rule yang berubah dan rule yang tetap gagal.
- Koreksi, pemulihan, pencegahan, serta acceptance criteria.
- Data Owner: mengonfirmasi definisi, threshold dan keputusan penggunaan.
- Data Steward: menjaga standar, mengoordinasikan issue dan bukti.
- Data Custodian: mengimplementasikan koreksi dan menjalankan ulang pipeline.

Pertanyaan debrief: bolehkah batch digunakan untuk training? Apa yang harus diselesaikan?
Data Docs membuktikan hasil aturan, belum membuktikan kelayakan AI secara menyeluruh.
Label quality, leakage, representativitas, dan schema checks memerlukan kontrol tambahan.
